# Phase 2: nhánh luật trên feature bộ phận × thời gian

Code ở `zeroshot/nesy/phase2.py` (mô hình), `features.py` (trích feature) và `phase1.py` (cách đánh giá dùng chung).

Mỗi split chạy 3 mô hình, cùng seed nên cùng tập validation:
- `emb`: nhánh embedding trên feature đã pool (giống phase 1).
- `rule1`: nhánh luật của phase 1 (feature đã pool).
- `rule2`: nhánh luật mới. Primitive được dự đoán theo bộ phận × thời gian, kèm luật thứ tự và ràng buộc loại trừ.

Sau đó ghép `emb + β·rule` sau khi train, với β chọn trên val. Mỗi kết quả có 2 cột:
- **clean**: epoch và γ chọn trên val.
- **best-on-test** (`_bt`): lấy max trên test, cách các paper báo cáo.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, gc, subprocess, importlib
import numpy as np, pandas as pd, torch

DRIVE_DIR   = '/content/drive/MyDrive/BaoHuy/zeroshot'
WEIGHTS_DIR = f'{DRIVE_DIR}/shiftgcn_weights'
SKEL = {'ntu60':  '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz',
        'ntu120': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU120_CSub.npz'}
TEXT_CACHE = f'{DRIVE_DIR}/phase1/text_proto.npz'    # tạo bởi phase1.ipynb (cell text)
OUT_DIR    = f'{DRIVE_DIR}/phase2'
# Cache feature bộ phận × thời gian (fp16): ~3.4 GB / split NTU60, ~7 GB / split NTU120.
# Mặc định để trên ổ local của Colab (mất khi hết phiên, trích lại ~45 phút cho 4 split).
# Drive còn chỗ thì đổi sang f'{DRIVE_DIR}/cache_p2' để lần sau khỏi trích.
CACHE_DIR  = '/content/cache_p2'
os.makedirs(OUT_DIR, exist_ok=True)

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
NESY = f'{REPO}/zeroshot/nesy'
if NESY not in sys.path:
    sys.path.insert(0, NESY)
import phase1, features, phase2
for m in (phase1, features, phase2):
    importlib.reload(m)
TASKS = list(phase1.UNSEEN)

In [ ]:
prims, names, R, imp = phase1.load_kg(f'{REPO}/zeroshot/kg_rules.json', f'{REPO}/zeroshot/primitive.json')
struct = phase2.load_kg_structure(f'{REPO}/zeroshot/kg_rules.json', f'{REPO}/zeroshot/primitive.json')
print(f'{len(prims)} primitive | {len(imp)} ràng buộc kéo theo | {len(struct["excl_pairs"])} cặp loại trừ')
print('nhóm loại trừ dùng:', struct['excl_used'])
print('bỏ qua (tay/cánh tay, hai bên có thể khác nhau):', struct['excl_skipped'])
print(f'{len(struct["seqs"])} lớp có luật thứ tự')
seq_cls = {c for c, _ in struct['seqs']}
for task in TASKS:
    u = phase1.UNSEEN[task]
    print(f'  {task}: lớp unseen có luật thứ tự:', [names[c] for c in u if c in seq_cls])

tp = np.load(TEXT_CACHE)
text_proto = tp['per_src'].mean(1)
text_proto /= np.linalg.norm(text_proto, axis=1, keepdims=True)
print('text_proto', text_proto.shape, list(tp['sources']))

In [ ]:
# Trích pooled (256) + cube (8 bộ phận × 16 thời gian × 256) bằng encoder Neuron đóng băng.
# Mỗi split dùng weights riêng (pretrain trên lớp seen của split đó). Có cache rồi thì bỏ qua.
importlib.reload(features)
for ds in ('ntu60', 'ntu120'):
    tasks = [t for t in TASKS if t.startswith(ds + '_')]
    if all(os.path.exists(features.cache_paths(CACHE_DIR, t, 'test')['label']) for t in tasks):
        print(ds, ': cache có sẵn'); continue
    skel = dict(np.load(SKEL[ds]))       # đọc một lần cho cả dataset (7-20 GB RAM)
    for t in tasks:
        features.extract_split(skel, WEIGHTS_DIR, t, phase1.UNSEEN[t], CACHE_DIR)
    del skel; gc.collect()
!du -sh {CACHE_DIR}

In [ ]:
for m in (phase1, phase2):
    importlib.reload(m)
SEEDS = [0]
TRAIN_KW = dict(epochs=30, warmup=5, lr=1e-3, wd=1e-4, bs=256)
POOLED_KW = dict(hidden=1024, dropout=0.3, beta_init=5.0, lam_prim=1.0, lam_cons=1.0)
PT_KW = dict(d=256, n_blocks=2, dropout=0.2, tau=1.0, beta_init=5.0, lam_prim=1.0, lam_cons=1.0, lam_excl=1.0)
ABLATE_SEQ = False                 # True: thêm rule2 không có luật thứ tự (ablation)
RES_CSV = f'{OUT_DIR}/results.csv'

results, hists, scores = [], {}, {}
def record(task, name, seed, out):
    res, hist, sc = out
    results.append(res)
    hists[(task, name, seed)], scores[(task, name, seed)] = hist, sc
    pd.DataFrame(results).to_csv(RES_CSV, index=False)

for task in TASKS:
    for seed in SEEDS:
        data = features.load_cache(CACHE_DIR, task, kind='pooled')
        for mode, name in (('emb', 'emb'), ('rule', 'rule1')):
            make = phase1.make_pooled(mode, text_proto, R, imp, **POOLED_KW)
            record(task, name, seed, phase1.run_split(data, task, make, name=name, seed=seed, train_kw=TRAIN_KW))
        del data
        data = features.load_cache(CACHE_DIR, task, kind='cube')
        variants = [('rule2', {})] + ([('rule2_noseq', dict(use_seq=False))] if ABLATE_SEQ else [])
        for name, extra in variants:
            make = phase2.make_parttime(R, imp, struct, **PT_KW, **extra)
            record(task, name, seed, phase1.run_split(data, task, make, name=name, seed=seed, train_kw=TRAIN_KW))
        del data; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(results)
show = ['task', 'model', 'seed', 'epoch_val', 'val_zsl', 'zsl', 'zsl_best_test', 'gzsl_H', 'H_best_test']
display(df[show].round(4))

In [ ]:
# Ghép emb + beta * rule (beta chọn trên val). So rule1 (phase 1) với rule2 (phase 2).
importlib.reload(phase1)
rows, tables = [], {}
rules = ['rule1', 'rule2'] + (['rule2_noseq'] if ABLATE_SEQ else [])
for task in TASKS:
    for seed in SEEDS:
        for rname in rules:
            chosen, table = phase1.posthoc_fusion(scores[(task, 'emb', seed)], scores[(task, rname, seed)])
            t = pd.DataFrame(table)
            tables[(task, rname, seed)] = t
            e0 = t[t.beta == 0].iloc[0]
            rows.append(dict(task=task, seed=seed, rule=rname, beta_val=chosen['beta'],
                             zsl_emb=e0.zsl, zsl_fused=chosen['zsl'], gain=chosen['zsl'] - e0.zsl,
                             zsl_emb_bt=e0.zsl_best_test, zsl_fused_bt=chosen['zsl_best_test'],
                             gain_bt=chosen['zsl_best_test'] - e0.zsl_best_test,
                             H_emb=e0.gzsl_H, H_fused=chosen['gzsl_H'],
                             H_emb_bt=e0.H_best_test, H_fused_bt=chosen['H_best_test']))
ph = pd.DataFrame(rows).round(4)
ph.to_csv(f'{OUT_DIR}/posthoc_fusion.csv', index=False)
print('_bt = best-on-test (cách các paper báo cáo); không hậu tố = clean (chọn trên val)')
display(ph)

In [ ]:
# Nét liền: val ZSL (chọn epoch). Nét đứt: test ZSL của lần train cuối.
import matplotlib.pyplot as plt
names_m = ['emb', 'rule1', 'rule2']
fig, axes = plt.subplots(1, len(TASKS), figsize=(4.5 * len(TASKS), 3.2))
for ax, task in zip(axes, TASKS):
    for i, n in enumerate(names_m):
        h = hists.get((task, n, SEEDS[0]))
        if h:
            ax.plot([r['epoch'] for r in h['val']], [r['val_zsl'] for r in h['val']], color=f'C{i}', label=n)
            ax.plot([r['epoch'] for r in h['test']], [r['test_zsl'] for r in h['test']], color=f'C{i}', ls='--')
    ax.set_title(task.replace('_xsub_', ' '), fontsize=9)
    ax.set_xlabel('epoch')
axes[0].set_ylabel('ZSL')
axes[0].legend()
plt.tight_layout()
plt.show()

## Đọc kết quả

- **`rule2` so với `rule1`** (bảng `run`, cột `zsl`): đây là cái lợi của feature bộ phận × thời gian và luật thứ tự đối với riêng nhánh luật.
- **`gain` của `rule2` so với của `rule1`** (bảng `posthoc`): đây là phần KG đóng góp thêm vào nhánh embedding. Đây là con số chính.
- **`_bt`**: số để đặt cạnh bảng của các paper. Trong luận văn ghi rõ hai cách tính.
- Muốn biết luật thứ tự giúp bao nhiêu thì đặt `ABLATE_SEQ = True`.

Gửi mình hai bảng `run` và `posthoc`.